# a2_report

> Phase 21 (`specs/multiagent/phase21/`): A2's measures and analysis. `p2Columns` scores a P2 board from its trace (Phase 15's `suppressed`, `suppressedHeld` over liar-first slots with an honest holder, E4's first wrong answers per honest agent-turn, Phase 19's `contestColumns`); `runP2Cell` and `runA2Cell` run P2 and A2 cells for `runWorkload`. `a2TeamEstimates` gives the per-team contrasts and P2's paired differences (no coverage check; Phase 22's `a2Forecast` uses it on the pilot); `a2Summaries` reads the frozen plan: Gate A2 (H21-1 with the manipulation check), H21-2 beside it, H21-3 with Holm, H21-4 and the core measures through Phase 19's `rl19_measures` and `rl19_spec`.

In [ ]:
#| default_exp a2_report

In [ ]:
#| hide
from nbdev.showdoc import *

## P2's boards

In [ ]:
#| export
import json

import numpy as np

from explRL.a2_cols import failureIds, successIds
from explRL.a2_grid import A2_PLAN, P2_N, manipulated, p2Grid
from explRL.a2_seq import _key, runBoard21Cell
from explRL.bench_stats import holm
from explRL.colony_report import bootSeeds, signFlipExact
from explRL.contest_eval import contestColumns
from explRL.liar_report import mechanics
from explRL.liar_trace import traceRun
from explRL.llm_runs import runLLMCell
from explRL.rl19_measures import index, lags, ratio, vals
from explRL.rl19_spec import _agents, dolIndex

NAN = float('nan')
_r = lambda a, b: a / b if b else NAN


def p2Columns(cfg, tr):
    """A traced P2 board's columns: Phase 15's mechanics (suppressed, contaminated, ...), suppressedHeld and the liar-first
    slots with and without an honest holder, first wrong answers per honest agent-turn, honest pooled share, Phase 19's columns."""
    b, nL = tr.board, tr.nLiar
    first, trueLater = {}, {(s[2], s[3]) for s in b.shareLog if s[1] >= nL and not s[6]}
    for s in b.shareLog:
        first.setdefault((s[2], s[3]), s)
    lf = [x for x, s in first.items() if s[1] < nL]
    held = [x for x in lf if any(h >= nL for h in b.tasks[x[0]].holders[x[1]])]
    fw = {(s[1], s[2]) for s in b.submits if s[1] >= nL and s[4] == 'wrong'}
    turns, bd = sum(1 for r in b.ledger if r[0] >= nL), b.pooledBound()
    return {**mechanics(tr.trace, b, nL), **contestColumns(cfg, b, [tr.log]), 'liarFirstHeld': len(held), 'liarFirstUnheld': len(lf) - len(held),
            'suppressedHeld': _r(sum(x not in trueLater for x in held), len(held)), 'firstWrongPerTurn': _r(len(fw), turns),
            'pooledBound': bd, 'honestPooled': _r(tr.rows[0]['welfareHonest'] * cfg.N, bd) if bd > 0 else NAN}


def runP2(cfg, seed, client):
    "runClue's signature: Phase 15's traced run (the same requests) with P2's columns."
    tr = traceRun(cfg, seed, client)
    return [{**tr.rows[0], **p2Columns(cfg, tr)}], tr.log


def runP2Cell(cell):
    return runLLMCell(cell, run=runP2)


def runA2Cell(cell):
    "For runWorkload: A2's sequence boards through runBoard21, P2's through runP2."
    return runBoard21Cell(cell) if cell['study'] == 'a2' else runP2Cell(cell)

## Per-team estimates

Guarding is read per identity (`idGuarding`), so the success and failure guards are found on every board whatever their seats. The contrast of a team and arm is the mean over boards 3-6 of the success pair's mean guarding minus the failure pair's; the gate statistic is own minus shared. P2's difference per size and seed is drop-only minus original `suppressedHeld`.

In [ ]:
#| export
def _ids(r, col):
    return json.loads(r[col]) if isinstance(r[col], str) else r[col]


def contrastOf(rs, team, boards=range(3, 7)):
    "Mean over `boards` of (success pair's mean guarding - failure pair's) from one team's rows of one arm."
    S, F, out = successIds(team), failureIds(team), []
    for r in rs:
        if _key(r)[2] in boards:
            g = np.array(_ids(r, 'idGuarding'), float)
            out.append(np.nanmean(g[S]) - np.nanmean(g[F]))
    return float(np.nanmean(out)) if out else NAN


def _p2key(r):
    return ('drop' if r['study'] == 'a2drop' else 'orig'), int(float(r['N'])), int(float(r['seed']))


def a2TeamEstimates(rows, p2rows=()):
    "{contrast: {arm: {team: v}}, gate: {team: own - shared}, p2: {N: {seed: drop - orig suppressedHeld}}} (no coverage check)."
    con = {}
    for t, a in sorted({_key(r)[:2] for r in rows}):
        con.setdefault(a, {})[t] = contrastOf([r for r in rows if _key(r)[:2] == (t, a)], t)
    gate = {t: con['own'][t] - con['shared'][t] for t in con.get('own', {}) if t in con.get('shared', {})}
    sh = {_p2key(r): float(r['suppressedHeld']) for r in p2rows}
    p2 = {N: {s: sh[('drop', N, s)] - sh[('orig', N, s)] for (k, n, s) in sh if k == 'drop' and n == N and ('orig', N, s) in sh} for N in P2_N}
    return dict(contrast=con, gate=gate, p2=p2)

## Summaries

`a2Summaries` refuses unless every planned board is there, except with `partial='stopped'` (a stop after wave 2): then it reports P2, the manipulation check and "Gate A2 not read". The core measures (FR-C5) are read through an index in `rl19_spec`'s layout: level `<arm>|seq16` on boards 3-6, `common|seq16` for boards 1-2 (reported once).

In [ ]:
#| export
def _persist(rs, nL, pairs):
    "rl19_spec.persistence's rule on one sequence: the correlation across agents of contest shares between boards p and p + 1."
    out = []
    for p in pairs:
        (t1, x1), (t2, x2) = _agents(rs, nL, (p,), 'agentContests'), _agents(rs, nL, (p + 1,), 'agentContests')
        ag = sorted(set(t1) & set(t2))
        a, b = [x1[i] / t1[i] for i in ag], [x2[i] / t2[i] for i in ag]
        if len(ag) > 2 and np.std(a) > 0 and np.std(b) > 0:
            out.append(float(np.corrcoef(a, b)[0, 1]))
    return float(np.mean(out)) if out else NAN


def core(ix, m, test, N):
    "The plan's core measures over a level's boards (rl19_measures' ratios; welfare per 1,000 tokens with the board's N)."
    rs, T = ix.get(f'{m}|{test}', []), [test]
    f = lambda r, k: float(r[k]) if r.get(k) not in (None, '') else NAN
    tok = sum(f(r, 'inputTokens') + f(r, 'outputTokens') for r in rs)
    net, lg, mean = sum(f(r, 'welfare') * N * f(r, 'tEnd') for r in rs), lags(ix, m, T), lambda k: float(np.mean(vals(ix, m, test, k) or [NAN]))
    return dict(gullibility=ratio(ix, m, T, 'misinformedFirst', 'honestTurns'), suppression=ratio(ix, m, T, 'uncorrected', 'liarFirst'),
                contestsPerLie=ratio(ix, m, T, 'contests', 'liarPosts'), timeToContest=float(np.median(lg)) if lg else NAN,
                guardedLies=ratio(ix, m, T, 'guardLies', 'liarPosts'), errorCorr=ratio(ix, m, T, 'sameWrong', 'wrongHonest'),
                collaboration=mean('sharePerHeld'), welfare=mean('welfare'), welfareHonest=mean('welfareHonest'),
                welfarePerKTok=_r(net, tok / 1000), tokensPerSolved=_r(tok, sum(vals(ix, m, test, 'solved'))), boards=len(rs))


def _boot(d, nboot):
    est, lo, hi = bootSeeds({(t, 16): v for t, v in d.items() if np.isfinite(v)}, nboot=nboot)
    return dict(estimate=est, lo=lo, hi=hi, n=len(d))


def a2Summaries(rows, p2rows, plan=A2_PLAN, partial=None, nboot=None):
    "The frozen analysis of A2 (see the module docstring); partial='stopped': P2, the manipulation check and 'Gate A2 not read'."
    nboot, teams, arms, al = nboot or plan['nboot'], plan['teams'], plan['arms'], plan['alpha']
    have, p2have = {_key(r) for r in rows}, {(_p2key(r)[0], *_p2key(r)[1:]) for r in p2rows}
    want = {(t, a, b) for t in teams for a in arms for b in range(1, plan['boards'] + 1) if partial != 'stopped' or b <= 2}
    p2want = {('drop' if c['study'] == 'a2drop' else 'orig', c['N'], c['seed']) for c in p2Grid(plan['p2']['seeds'])}
    if want - have or p2want - p2have:
        raise ValueError(f'missing boards: {sorted(want - have)[:5]} P2 {sorted(p2want - p2have)[:5]} ({len(want - have)}, {len(p2want - p2have)})')
    E, own = a2TeamEstimates(rows, p2rows), [r for r in rows if r['arm'] == 'own']
    man = {t: manipulated(own, t) for t in teams}
    read = sum(man.values()) >= plan['manipulation']['teams']
    p3 = {N: signFlipExact([-x for x in E['p2'][N].values()]) for N in P2_N}
    adj = dict(zip(P2_N, holm([p3[N] for N in P2_N])))
    p2ix = index([dict(r, level=f"{_p2key(r)[0]}|p2N{_p2key(r)[1]}") for r in p2rows])
    P2 = {N: dict(diff=_boot(E['p2'][N], nboot), p=p3[N], pHolm=float(adj[N]), supported=bool(adj[N] <= al),
                  **{k: {m: float(np.mean(vals(p2ix, m, f'p2N{N}', c) or [NAN])) for m in ('orig', 'drop')}
                     for k, c in (('suppressedHeld', 'suppressedHeld'), ('suppressed', 'suppressed'), ('firstWrongPerTurn', 'firstWrongPerTurn'),
                                  ('contaminated', 'contaminated'), ('honestPooled', 'honestPooled'), ('sharePerHeld', 'sharePerHeld'))},
                  core={m: core(p2ix, m, f'p2N{N}', N) for m in ('orig', 'drop')}) for N in P2_N}
    pooled = [x for N in P2_N for x in E['p2'][N].values()]
    out = dict(manipulation=dict(perTeam=man, passed=sum(man.values()), of=len(teams), read=read), p2=P2,
               p2Pooled=dict(diff=float(np.mean(pooled)) if pooled else NAN, n=len(pooled)), notMeasured=plan['notMeasured'])
    ix12 = index([dict(r, level='common|seq16') for r in own if _key(r)[2] <= 2])
    out['boards12'] = dict(core(ix12, 'common', 'seq16', 16), liarOnFailure=int(sum(float(r['liarOnFailure']) for r in own if _key(r)[2] <= 2)))
    if partial == 'stopped':
        return dict(out, gate=dict(read=False, passes=False, note='Gate A2 not read: the run stopped after wave 2'))
    h1, h2 = signFlipExact(list(E['gate'].values())), signFlipExact(list(E['contrast']['own'].values()))
    ix = index([dict(r, level=f"{r['arm']}|seq16") for r in rows if _key(r)[2] >= 3])
    seqs = lambda a, t: [r for r in rows if _key(r)[:2] == (t, a)]
    per = lambda a, k, lo=3: float(np.nanmean([float(r[k]) for r in rows if r['arm'] == a and _key(r)[2] >= lo]))
    arm = {a: dict(contrast=_boot(E['contrast'][a], nboot), pVs0=signFlipExact(list(E['contrast'][a].values())),
                   dol=_boot({t: dolIndex(seqs(a, t), 4, positions=range(2, 6)) for t in teams}, nboot),
                   persistence=_boot({t: _persist(seqs(a, t), 4, range(5)) for t in teams}, nboot),
                   persistence36=_boot({t: _persist(seqs(a, t), 4, (2, 3, 4)) for t in teams}, nboot),
                   volunteers=per(a, 'volunteers'), specExcess=per(a, 'specIndex') - per(a, 'specNull'),
                   guarding={b: [float(np.nanmean([np.nanmean(np.array(_ids(r, 'idGuarding'), float)[ids(r)]) for r in rows if r['arm'] == a and _key(r)[2] == b]))
                                 for ids in (lambda r: successIds(_key(r)[0]), lambda r: failureIds(_key(r)[0]))] for b in range(1, plan['boards'] + 1)},
                   core=core(ix, a, 'seq16', 16)) for a in arms}
    passes = bool(h1 <= al and read)
    note = ('records changed guarding but not as predicted' if passes and not h2 <= al else
            None if read else 'Gate A2 not read: the manipulation check failed')
    return dict(out, gate=dict(read=read, passes=passes, H21_1=dict(p=h1, perTeam=E['gate'], est=_boot(E['gate'], nboot)),
                               H21_2=dict(p=h2, supported=bool(h2 <= al)), note=note), arms=arm)

## Tests

In [ ]:
# test_a2_measures (AC-10, FR-C5): on a hand-built 6-board team whose per-slot columns are seated by seating(team, b),
# dolIndex over boards 3-6 and the persistence rule equal hand counts made per identity
from explRL.a2_cols import seating
from explRL.a2_report import *
from explRL.a2_report import _persist

team, rng = 2100, np.random.default_rng(5)
tasks = {b: rng.integers(1, 9, size=(12, 5)) for b in range(1, 7)}      # per identity: contest, other share, submit, open, wait
rs = []
for b in range(1, 7):
    seat, T = seating(team, b), tasks[b]
    slot = lambda f: json.dumps([0] * 4 + [int(f(T[a])) for a in seat])
    rs.append(dict(pos=b - 1, seqId=team, nLiar=4, agents=json.dumps([0, 1, 2, 3] + [a + 4 for a in seat]),
                   agentTurns=slot(lambda t: t.sum()), agentShares=slot(lambda t: t[0] + t[1]), agentContests=slot(lambda t: t[0]),
                   agentSubmits=slot(lambda t: t[2]), agentOpens=slot(lambda t: t[3])))
M = sum(tasks[b] for b in range(3, 7)).astype(float)
p = M / M.sum()
pa, pt = p.sum(1, keepdims=True), p.sum(0)
H = -(pt * np.log(pt)).sum()
assert abs(dolIndex(rs, 4, positions=range(2, 6)) - (p * np.log(p / (pa * pt))).sum() / H) < 1e-12
share = {b: tasks[b][:, 0] / tasks[b].sum(1) for b in range(1, 7)}
hand = lambda pairs: np.mean([np.corrcoef(share[b], share[b + 1])[0, 1] for b in pairs])
assert abs(_persist(rs, 4, range(5)) - hand(range(1, 6))) < 1e-12 and abs(_persist(rs, 4, (2, 3, 4)) - hand((3, 4, 5))) < 1e-12

In [ ]:
# p2Columns (FR-C2) on traced boards (rule agents, Phase 14's board with N // 4 liars): suppressedHeld counts only
# liar-first slots with an honest holder, and equals Phase 15's suppressed on the board where every liar-first slot has
# one (no such board occurs in seeds 900-1099 at N 4 or 8: built by dropping the posts on slots no honest agent holds);
# runA2Cell sends A2 cells to runBoard21 and P2 cells to runP2 (the fake client)
import copy
import tempfile
from explRL.a2_grid import boardCell21, p2Cell
from explRL.liar_report import mechanics
from explRL.llm_runs import LLMConfig, makeClient

seen = set()
for seed in range(900, 960):
    c = p2Cell(8, seed, False)
    cfg = LLMConfig.fromDict(c['cfg'])
    tr = traceRun(cfg, seed)
    col, b = p2Columns(cfg, tr), tr.board
    first = {}
    for s in b.shareLog:
        first.setdefault((s[2], s[3]), s)
    lf = [x for x, s in first.items() if s[1] < 2]
    held = [x for x in lf if any(h >= 2 for h in b.tasks[x[0]].holders[x[1]])]
    later = {(s[2], s[3]) for s in b.shareLog if s[1] >= 2 and not s[6]}
    assert col['liarFirstHeld'] == len(held) and col['liarFirstUnheld'] == len(lf) - len(held)
    if held:
        assert col['suppressedHeld'] == sum(x not in later for x in held) / len(held)
    b2 = copy.copy(b)
    b2.shareLog = [s for s in b.shareLog if any(h >= 2 for h in b.tasks[s[2]].holders[s[3]])]
    m2 = mechanics(tr.trace, b2, 2)['suppressed']
    assert (col['suppressedHeld'] == m2) or (col['suppressedHeld'] != col['suppressedHeld'] and m2 != m2)
    assert p2Columns(cfg, tr._replace(board=b2))['liarFirstUnheld'] == 0
    if col['liarFirstUnheld'] and held:
        seen.add('some unheld')
assert seen == {'some unheld'}, seen
with tempfile.TemporaryDirectory() as d:
    for c in (p2Cell(8, 900, True), boardCell21(2100, 1, None)):
        c = dict(c, cfg=dict(c['cfg'], turns=2), client=dict(kind='fake', cache=d))
        r = runA2Cell(c)[0]
        assert ('suppressedHeld' in r) == (c['study'] == 'a2drop') and ('idGuarding' in r) == (c['study'] == 'a2')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()